# Where EVA02 looks: OOF attribution maps

Visualize `interp/` methods on identity-disjoint fold-0 OOF of LLM2CLIP EVA02-L-14-336 (`runs/cv/eva02_trial23`).

The retrieval embedding is **attention-pooled patch tokens**, not the CLS token. `pooling` is therefore the map the model actually uses. Transformer methods (`last_attn`, `rollout`, `chefer`) still show how CLS attends. CAM methods attribute the **pre-normalization neck vector** — L2 energy of the unit embedding is constant, so Grad-CAM on it is empty. For query–gallery pairs we pass the other image's embedding as `reference` so the map explains cosine similarity.


In [ ]:
import json
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "train.csv").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from hydra import compose, initialize_config_dir
from PIL import Image

from augmentations import build_transforms
from dataset.images import crop_bbox, image_path
from eval import load_model
from interp import METHODS, interpret, normalize_maps, overlay
from postproc.retrieval import normalize

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})


def display(obj):
    print(obj if isinstance(obj, str) else obj.to_string() if hasattr(obj, "to_string") else obj)


CV = ROOT / "runs" / "cv" / "eva02_trial23"
IMAGES = ROOT / "data" / "images"
FOLD = 0
VAL = CV / f"fold{FOLD}" / "val"


def pick_device():
    if not torch.cuda.is_available():
        return torch.device("cpu")
    for index in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(index)
        if free / max(total, 1) > 0.95:
            return torch.device(f"cuda:{index}")
    return torch.device("cpu")


DEVICE = pick_device()
print("ROOT", ROOT)
print("device", DEVICE)

Please 'pip install xformers'
Please 'pip install xformers'
ROOT /data/projects/ryzhichkin/ReID
device cuda:0


/data/projects/ryzhichkin/ReID/.venv/lib/python3.11/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error timed out>
  data = fetch_version_info()
/data/projects/ryzhichkin/ReID/.venv/lib/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
metrics = json.loads((VAL / "metrics.json").read_text())
ckpt = metrics["checkpoint"]
with initialize_config_dir(version_base="1.3", config_dir=str(ROOT / "configs")):
    cfg = compose(
        config_name="config",
        overrides=[
            f"checkpoint={ckpt}",
            "experiment=current_best_tuned",
            f"eval.device={DEVICE}",
            "eval.split=val",
            "eval.weights=ema",
            "eval.precision=fp32",
            f"eval.output_dir={VAL}",
            "eval.top_k=10",
            "eval.save_distances=false",
        ],
    )
model, cfg, checkpoint, weights = load_model(cfg, override_items=[])
model.to(DEVICE).eval()
CONTEXT = float(cfg.data.context_pct)
print("weights", weights, "pooling", cfg.model.pooling.kind, "size", list(cfg.data.image_size))

weights ema pooling attn size [336, 336]


In [3]:
oof = pd.read_csv(VAL / "oof.csv", dtype={"image_id": str})
query = pd.read_csv(VAL / "query.csv", dtype={"image_id": str})
gallery = pd.read_csv(VAL / "gallery.csv", dtype={"image_id": str})
emb = normalize(np.load(VAL / "embeddings.npy"))
lookup = {image_id: i for i, image_id in enumerate(oof.image_id)}
qe = emb[[lookup[i] for i in query.image_id]]
ge = emb[[lookup[i] for i in gallery.image_id]]
dist = 1.0 - qe @ ge.T
rows = []
for i in range(len(query)):
    keep = gallery.camera_id.to_numpy() != query.camera_id.iloc[i]
    order = np.argsort(dist[i], kind="stable")
    order = order[keep[order]]
    relevant = gallery.vehicle_id.to_numpy()[order] == query.vehicle_id.iloc[i]
    found = np.flatnonzero(relevant)
    rows.append(
        {
            "q_index": i,
            "image_id": query.image_id.iloc[i],
            "vehicle_id": int(query.vehicle_id.iloc[i]),
            "ap": float(np.mean((np.arange(len(found)) + 1) / (found + 1))),
            "rank1_ok": bool(relevant[0]),
            "r1_id": gallery.image_id.iloc[order[0]],
            "pos_id": gallery.image_id.iloc[order[found[0]]],
            "r1_vehicle": int(gallery.vehicle_id.iloc[order[0]]),
        }
    )
qtab = pd.DataFrame(rows)
hard = qtab.sort_values("ap").head(2)
easy = qtab.sort_values("ap", ascending=False).head(2)
miss = qtab[~qtab.rank1_ok].head(2)
picked = pd.concat([easy, hard, miss]).drop_duplicates("image_id").head(6)
print(qtab.ap.describe())
display(picked[["image_id", "vehicle_id", "ap", "rank1_ok", "r1_vehicle"]])

count    309.000000
mean       0.858304
std        0.269616
min        0.037669
25%        0.855159
50%        1.000000
75%        1.000000
max        1.000000
Name: ap, dtype: float64
                             image_id  vehicle_id        ap  rank1_ok  r1_vehicle
308  f8100578c7344b06acd040ae4ea6543a        1283  1.000000      True        1283
0    34ce7989a4c440a882c40fa7891ce9d4          63  1.000000      True          63
266  54773bbcb4614ed7aa93e20744e79cbc        1117  0.037669     False        1067
63   fc7507f7b2a649a99b670e047a94526a         231  0.046401     False         411
1    51959ce2fd8e4f98aa2438851df9cbcf        1005  0.241604     False         135
30   019df33848224006bf565042c625c773          58  0.135450     False         429


In [4]:
transform = build_transforms(cfg)


def frame_row(image_id):
    hit = oof.loc[oof.image_id == image_id]
    if len(hit) == 0:
        hit = pd.concat([query, gallery], ignore_index=True)
        hit = hit.loc[hit.image_id == image_id]
    return hit.iloc[0]


def load_crop_array(image_id):
    row = frame_row(image_id)
    with Image.open(image_path(IMAGES, image_id)) as im:
        crop = crop_bbox(im.convert("RGB"), [row.x, row.y, row.w, row.h], CONTEXT)
    return np.asarray(crop.convert("RGB")), row


def model_tensor(image_id):
    rgb, row = load_crop_array(image_id)
    tensor = transform(rgb)
    return tensor.unsqueeze(0).to(DEVICE), rgb, row


def explain(image_id, method, reference=None):
    tensor, rgb, _ = model_tensor(image_id)
    ref = None if reference is None else torch.as_tensor(reference, device=DEVICE)
    with torch.enable_grad():
        heatmap = interpret(model, tensor, method=method, reference=ref)[0]
    canvas = overlay(np.array(Image.fromarray(rgb).resize(tensor.shape[-2:][::-1])), heatmap)
    return rgb, heatmap, canvas


print("methods", list(METHODS))

methods ['pooling', 'last_attn', 'rollout', 'chefer', 'gradcam', 'hirescam', 'layercam', 'eigencam']


## All methods on the same queries

Each row is one OOF query. Columns are the eight `interp` methods. Warm colors = higher attribution.


In [5]:
cases = picked.to_dict("records")
method_names = list(METHODS)
stats = []
fig, axes = plt.subplots(len(cases), 1 + len(method_names), figsize=(18, 2.4 * len(cases)))
for r, rec in enumerate(cases):
    rgb, _ = load_crop_array(rec["image_id"])
    axes[r, 0].imshow(rgb)
    tag = "R1 ok" if rec["rank1_ok"] else "R1 miss"
    axes[r, 0].set_title(f"id {rec['vehicle_id']} AP {rec['ap']:.2f}\n{tag}", fontsize=8)
    axes[r, 0].axis("off")
    for c, name in enumerate(method_names, start=1):
        _, heatmap, canvas = explain(rec["image_id"], name)
        normed = normalize_maps(heatmap)[0]
        stats.append(
            {
                "image_id": rec["image_id"],
                "method": name,
                "raw_std": float(heatmap.std()),
                "norm_std": float(normed.std()),
                "finite": bool(np.isfinite(heatmap).all()),
            }
        )
        axes[r, c].imshow(canvas)
        if r == 0:
            axes[r, c].set_title(name, fontsize=9)
        axes[r, c].axis("off")
plt.tight_layout()
FIG_DIR = ROOT / "notebooks" / "eva02" / "interp_figs"
FIG_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG_DIR / "all_methods.png", dpi=120, bbox_inches="tight")
plt.show()
stat = pd.DataFrame(stats)
display(stat.groupby("method")[["raw_std", "norm_std"]].mean())
bad = stat[~stat.finite | (stat["norm_std"] < 0.05)]
print("low-contrast maps", len(bad))
if len(bad):
    display(bad)
if not stat.finite.all():
    raise RuntimeError("Non-finite interpretation maps")

               raw_std  norm_std
method                          
chefer     4224.983561  0.127795
eigencam      0.010869  0.174219
gradcam       0.176901  0.213306
hirescam      1.652067  0.141043
last_attn     0.002688  0.092317
layercam      0.842367  0.142153
pooling       0.003201  0.150026
rollout       0.000465  0.134767
low-contrast maps 0


Saved figure (also written to `notebooks/eva02/interp_figs/all_methods.png`):

![all methods](interp_figs/all_methods.png)


## Cosine to the true positive vs cosine to Rank-1

Grad-CAM / Chefer / pooling with `reference=` the other embedding. If Rank-1 is wrong, the distractor map shows which parts of the query made the model prefer the lookalike.


In [6]:
pair_methods = ("pooling", "gradcam", "chefer")
pair_rows = pd.concat([easy.head(1), miss.head(1)]).to_dict("records")
fig, axes = plt.subplots(len(pair_rows), 1 + 3 * len(pair_methods), figsize=(16, 3.2 * len(pair_rows)))
if len(pair_rows) == 1:
    axes = np.array([axes])
for r, rec in enumerate(pair_rows):
    q_rgb, _ = load_crop_array(rec["image_id"])
    p_rgb, _ = load_crop_array(rec["pos_id"])
    d_rgb, _ = load_crop_array(rec["r1_id"])
    q_emb = torch.as_tensor(qe[rec["q_index"]], device=DEVICE)
    p_emb = torch.as_tensor(emb[lookup[rec["pos_id"]]], device=DEVICE)
    d_emb = torch.as_tensor(emb[lookup[rec["r1_id"]]], device=DEVICE)
    axes[r, 0].imshow(q_rgb)
    axes[r, 0].set_title(f"query id {rec['vehicle_id']}", fontsize=8)
    axes[r, 0].axis("off")
    col = 1
    for name in pair_methods:
        _, _, to_pos = explain(rec["image_id"], name, reference=p_emb)
        _, _, to_neg = explain(rec["image_id"], name, reference=d_emb)
        _, _, on_pos = explain(rec["pos_id"], name, reference=q_emb)
        for canvas, title in (
            (to_pos, f"{name}\nquery→+"),
            (on_pos, f"{name}\ngallery+→q"),
            (to_neg, f"{name}\nquery→R1"),
        ):
            axes[r, col].imshow(canvas)
            axes[r, col].set_title(title, fontsize=8)
            axes[r, col].axis("off")
            col += 1
plt.tight_layout()
FIG_DIR = ROOT / "notebooks" / "eva02" / "interp_figs"
FIG_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG_DIR / "query_gallery.png", dpi=120, bbox_inches="tight")
plt.show()
print(
    "left-to-right per method: query explained by the true positive,"
    " the positive explained by the query, query explained by Rank-1"
)

left-to-right per method: query explained by the true positive, the positive explained by the query, query explained by Rank-1


Saved figure (also written to `notebooks/eva02/interp_figs/query_gallery.png`):

![query vs gallery](interp_figs/query_gallery.png)


## How to read this

- **`pooling`**: trained spatial attention over patches. This is what is summed into the 256-D embedding.
- **`gradcam` / `hirescam` / `layercam`**: gradient of the embedding (or of cosine to a gallery vector) w.r.t. the last ViT tokens, reshaped to 24×24.
- **`eigencam`**: dominant activation direction, no labels. Useful as a gradient-free sanity check.
- **`last_attn` / `rollout` / `chefer`**: CLS attention. They can disagree with `pooling` because this recipe does not retrieve with CLS.
- Rank-1 misses usually highlight the same body panel / color region on query and distractor — the model is looking at the vehicle, just the wrong identity.
